# logit-thermo A2-19：A2-18 の XQ1（予言した罰 h*）を、新しい BBC の記事 120 本でもう一度だけ確かめる——ランタイム 2/3（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 42 個の仕事（7 モデル × 2 つの腕（罰なしの対照・予言した罰 h*）× 記事の 3 つの組）を、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行きます。3 つ同時なら 7 時間くらいの見込みです（CU は 35 くらい）。4 本目以降のランタイムで同じノートブックを開いても、仕事を取った印はランタイムごとに別なので、ふつうは重なりません（ほぼ同時に始めると、まれに同じ仕事を 2 回回すことがあります。中身は同じになるので、結果は変わりません）。
- 作り：A2-18 と同じ（罰の強さ h* と温度は A2-18 のまま、サンプル 4、512 歩、q つき）。記事だけが新しい：A2-2 と同じ月（2025-04〜06）の BBC の記事から、これまでの 40 本を除いて 120 本を種で選びます。
- **データは、ゲートを凍結するまで開けません。** 結果も記事の中身も表示しません。
- 計算のコード：これまでのデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_18_collect.py`（`0b9d203ebe0a71e6…`）と `scripts/a2_19_collect.py`（`de4d86520b5ab6a4…`）は、Drive の `A2_explore/` の .py から読んでハッシュで確かめる。
- 保存先 `A2_19/`。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします。
- **通知**：🚀・仕事ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-19 罰の強さの予言・もう一度だけ（新しい記事 120 本 × 7 モデル × 対照・h*、ランタイム 2/3、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-19 罰の強さの予言・もう一度だけ（新しい記事 120 本 × 7 モデル × 対照・h*、ランタイム 2/3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-19 penalty replication worker 2of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', 'SRCG': '49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCD2, SRCG = _s['SRCD2'], _s['SRCG']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
def take_file(name, want):
    """Drive の logit-thermo のフォルダから、名前とハッシュが合う .py のファイルを読む。"""
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/' + name, recursive=True)):
        s = open(p, encoding='utf-8').read()
        if hashlib.sha256(s.encode()).hexdigest() == want:
            return s
    raise AssertionError(('Drive に ' + name + ' （ハッシュが合うもの）が見つかりません', want))
SRC17 = take_file('a2_18_collect.py', '0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496')
SRC19 = take_file('a2_19_collect.py', 'de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef')
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
exec(SRC17)
exec(SRC19)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → BBC の記事（A2-2 と同じか確かめる）→ 新しい 120 本 → 仕事を取りに行って回す → 保存（結果も記事も表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_gpre(), self_test_a18(), self_test_a19())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_19', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    GROUPS, META19 = load_docs19('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json')
    import glob as _glob
    for mp in sorted(_glob.glob('/content/drive/MyDrive/logit-thermo/A2_19' + '/docs_meta_w*.json')):
        prev = json.load(open(mp))
        assert [[d['text_sha256'] for d in g] for g in prev['groups']] == [[d['text_sha256'] for d in g] for g in META19['groups']], ('新しい記事がほかのランタイムと違います', mp)
    json.dump(META19, open('/content/drive/MyDrive/logit-thermo/A2_19' + '/docs_meta_w1.json', 'w'), ensure_ascii=False, indent=1)
    N_DONE, LEFT = run_a19(1, 3, GROUPS, '/content/drive/MyDrive/logit-thermo/A2_19', 'cuda', notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=1, done=N_DONE, left=LEFT, samples=GP_S, chunk=GP_CHUNK,
                   a19_sha256='de4d86520b5ab6a42f02a7dea8a3048a943711ba26499d3a08ca061aad27b3ef', a18_sha256='0b9d203ebe0a71e67d3c419897212c2091f04697c0187ac9475f9112eeea6496', gpre_sha256='49aa63e0c3479f9841bf4b17740325fe21ddaea8b118f03730461959ef03e44b', dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_19' + '/run_meta_worker1.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-19 罰の強さの予言・もう一度だけ（ランタイム 2/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：A2-19 罰の強さの予言・もう一度だけ（ランタイム 2/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
